# MLMD Lab 1: Tracking a Sports Analytics ML Pipeline with ML Metadata

**Author:** Tanish Dhongade  
**Course:** IE 7374 - Machine Learning Operations (MLOps)  

---

## Overview

This notebook demonstrates how to use [ML Metadata (MLMD)](https://www.tensorflow.org/tfx/guide/mlmd) to track metadata across an end-to-end sports analytics ML pipeline. Instead of the generic training example from the original lab, the entire workflow is framed around **predicting football (soccer) player market valuations** using match event data.

### Modifications from Original Lab
| Aspect | Original Lab | This Version |
|---|---|---|
| Domain | Generic ML training | Football / soccer player valuation |
| Artifact types | DataSet, SavedModel | MatchEventData, PlayerFeatureSet, ValuationModel, EvaluationMetrics |
| Pipeline steps | Single trainer | Ingest -> Feature Eng -> Train -> Evaluate (4 steps) |
| Context | Single experiment | Season-based experiment grouping |
| Querying | Basic retrieval | Lineage tracing + conditional filtering |

In [3]:
!apt-get update -qq && apt-get install -y -qq python3.10 python3.10-venv python3.10-dev 2>&1 | tail -1
!python3.10 -m ensurepip --upgrade 2>&1 | tail -1
!python3.10 -m pip install ml-metadata --quiet 2>&1 | tail -3
!python3.10 -c "from ml_metadata import metadata_store; print('SUCCESS:', dir(metadata_store))"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Processing triggers for man-db (2.10.2-1) ...

SUCCESS: ['ListOptions', 'MetadataStore', 'OrderByField', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'downgrade_schema', 'metadata_store', 'pywrap']


## Pipeline Walkthrough

The cell below runs the full MLMD pipeline end-to-end using Python 3.10 (required since `ml-metadata` does not support Python 3.12). It covers:

1. **Setup** - Initialise an in-memory SQLite metadata store
2. **Register Types** - 4 artifact types (MatchEventData, PlayerFeatureSet, ValuationModel, EvaluationMetrics) and 4 execution types (DataIngestor, FeatureEngineer, ModelTrainer, ModelEvaluator)
3. **Pipeline Run 1** - Ingest Bundesliga 2023-24 data, engineer per-90 features, train XGBoost model (lr=0.05, 300 trees), evaluate (R2=0.847)
4. **Context Grouping** - Link all artifacts and executions to a SeasonExperiment context
5. **Querying** - Retrieve artifacts by type, trace full upstream lineage, run conditional filter queries
6. **Pipeline Run 2** - Updated feature set (v2.0, 58 features with progressive carries + pressing), retrain (lr=0.03, 500 trees), evaluate (R2=0.912)
7. **Experiment Comparison** - Compare both runs side-by-side through MLMD context queries

In [4]:
%%script python3.10

import ml_metadata as mlmd
from ml_metadata import metadata_store
from ml_metadata.proto import metadata_store_pb2

print("=" * 60)
print("1. SETUP")
print("=" * 60)
print("ml-metadata imported successfully.")

# ============================================================
# 2. Create the Metadata Store
# ============================================================
connection_config = metadata_store_pb2.ConnectionConfig()
connection_config.fake_database.SetInParent()
store = metadata_store.MetadataStore(connection_config)
print("\nMetadata store initialised successfully (in-memory SQLite backend).")

# ============================================================
# 3. Register Artifact Types
# ============================================================
print("\n" + "=" * 60)
print("3. REGISTER ARTIFACT TYPES")
print("=" * 60)

match_data_type = metadata_store_pb2.ArtifactType()
match_data_type.name = "MatchEventData"
match_data_type.properties["season"] = metadata_store_pb2.STRING
match_data_type.properties["league"] = metadata_store_pb2.STRING
match_data_type.properties["num_matches"] = metadata_store_pb2.INT
match_data_type_id = store.put_artifact_type(match_data_type)
print(f"Registered ArtifactType 'MatchEventData' with id={match_data_type_id}")

feature_set_type = metadata_store_pb2.ArtifactType()
feature_set_type.name = "PlayerFeatureSet"
feature_set_type.properties["num_players"] = metadata_store_pb2.INT
feature_set_type.properties["num_features"] = metadata_store_pb2.INT
feature_set_type.properties["feature_version"] = metadata_store_pb2.STRING
feature_set_type_id = store.put_artifact_type(feature_set_type)
print(f"Registered ArtifactType 'PlayerFeatureSet' with id={feature_set_type_id}")

model_type = metadata_store_pb2.ArtifactType()
model_type.name = "ValuationModel"
model_type.properties["model_type"] = metadata_store_pb2.STRING
model_type.properties["version"] = metadata_store_pb2.INT
model_type.properties["framework"] = metadata_store_pb2.STRING
model_type_id = store.put_artifact_type(model_type)
print(f"Registered ArtifactType 'ValuationModel' with id={model_type_id}")

eval_type = metadata_store_pb2.ArtifactType()
eval_type.name = "EvaluationMetrics"
eval_type.properties["rmse"] = metadata_store_pb2.DOUBLE
eval_type.properties["mae"] = metadata_store_pb2.DOUBLE
eval_type.properties["r2_score"] = metadata_store_pb2.DOUBLE
eval_type_id = store.put_artifact_type(eval_type)
print(f"Registered ArtifactType 'EvaluationMetrics' with id={eval_type_id}")

registered_types = store.get_artifact_types()
print(f"\nTotal registered artifact types: {len(registered_types)}")
for t in registered_types:
    print(f"  - {t.name} (id={t.id}): properties={dict(t.properties)}")

# ============================================================
# 4. Register Execution Types
# ============================================================
print("\n" + "=" * 60)
print("4. REGISTER EXECUTION TYPES")
print("=" * 60)

ingestor_type = metadata_store_pb2.ExecutionType()
ingestor_type.name = "DataIngestor"
ingestor_type.properties["state"] = metadata_store_pb2.STRING
ingestor_type.properties["source_format"] = metadata_store_pb2.STRING
ingestor_type_id = store.put_execution_type(ingestor_type)
print(f"Registered ExecutionType 'DataIngestor' with id={ingestor_type_id}")

fe_type = metadata_store_pb2.ExecutionType()
fe_type.name = "FeatureEngineer"
fe_type.properties["state"] = metadata_store_pb2.STRING
fe_type.properties["aggregation_method"] = metadata_store_pb2.STRING
fe_type_id = store.put_execution_type(fe_type)
print(f"Registered ExecutionType 'FeatureEngineer' with id={fe_type_id}")

trainer_type = metadata_store_pb2.ExecutionType()
trainer_type.name = "ModelTrainer"
trainer_type.properties["state"] = metadata_store_pb2.STRING
trainer_type.properties["learning_rate"] = metadata_store_pb2.DOUBLE
trainer_type.properties["n_estimators"] = metadata_store_pb2.INT
trainer_type_id = store.put_execution_type(trainer_type)
print(f"Registered ExecutionType 'ModelTrainer' with id={trainer_type_id}")

evaluator_type = metadata_store_pb2.ExecutionType()
evaluator_type.name = "ModelEvaluator"
evaluator_type.properties["state"] = metadata_store_pb2.STRING
evaluator_type.properties["eval_split"] = metadata_store_pb2.STRING
evaluator_type_id = store.put_execution_type(evaluator_type)
print(f"Registered ExecutionType 'ModelEvaluator' with id={evaluator_type_id}")

registered_exec_types = store.get_execution_types()
print(f"\nTotal registered execution types: {len(registered_exec_types)}")
for t in registered_exec_types:
    print(f"  - {t.name} (id={t.id}): properties={dict(t.properties)}")

# ============================================================
# 5. Pipeline Run 1: Bundesliga 2023-24
# ============================================================
print("\n" + "=" * 60)
print("5. PIPELINE RUN 1: BUNDESLIGA 2023-24")
print("=" * 60)

# 5.1 Data Ingestion
print("\n--- 5.1 Data Ingestion ---")
raw_data = metadata_store_pb2.Artifact()
raw_data.uri = "gs://sports-ml-data/bundesliga/2023-24/match_events.parquet"
raw_data.type_id = match_data_type_id
raw_data.properties["season"].string_value = "2023-24"
raw_data.properties["league"].string_value = "Bundesliga"
raw_data.properties["num_matches"].int_value = 306
[raw_data_id] = store.put_artifacts([raw_data])
print(f"Created MatchEventData artifact (id={raw_data_id})")

ingest_exec = metadata_store_pb2.Execution()
ingest_exec.type_id = ingestor_type_id
ingest_exec.properties["state"].string_value = "RUNNING"
ingest_exec.properties["source_format"].string_value = "parquet"
[ingest_exec_id] = store.put_executions([ingest_exec])
print(f"Created DataIngestor execution (id={ingest_exec_id})")

ingest_output_event = metadata_store_pb2.Event()
ingest_output_event.artifact_id = raw_data_id
ingest_output_event.execution_id = ingest_exec_id
ingest_output_event.type = metadata_store_pb2.Event.DECLARED_OUTPUT
store.put_events([ingest_output_event])

ingest_exec.id = ingest_exec_id
ingest_exec.properties["state"].string_value = "COMPLETED"
store.put_executions([ingest_exec])
print("Data ingestion step completed.")

# 5.2 Feature Engineering
print("\n--- 5.2 Feature Engineering ---")
fe_exec = metadata_store_pb2.Execution()
fe_exec.type_id = fe_type_id
fe_exec.properties["state"].string_value = "RUNNING"
fe_exec.properties["aggregation_method"].string_value = "per_90_minutes"
[fe_exec_id] = store.put_executions([fe_exec])
print(f"Created FeatureEngineer execution (id={fe_exec_id})")

fe_input_event = metadata_store_pb2.Event()
fe_input_event.artifact_id = raw_data_id
fe_input_event.execution_id = fe_exec_id
fe_input_event.type = metadata_store_pb2.Event.DECLARED_INPUT
store.put_events([fe_input_event])

player_features = metadata_store_pb2.Artifact()
player_features.uri = "gs://sports-ml-data/bundesliga/2023-24/player_features_v1.parquet"
player_features.type_id = feature_set_type_id
player_features.properties["num_players"].int_value = 523
player_features.properties["num_features"].int_value = 42
player_features.properties["feature_version"].string_value = "v1.0"
[player_features_id] = store.put_artifacts([player_features])
print(f"Created PlayerFeatureSet artifact (id={player_features_id})")

fe_output_event = metadata_store_pb2.Event()
fe_output_event.artifact_id = player_features_id
fe_output_event.execution_id = fe_exec_id
fe_output_event.type = metadata_store_pb2.Event.DECLARED_OUTPUT
store.put_events([fe_output_event])

fe_exec.id = fe_exec_id
fe_exec.properties["state"].string_value = "COMPLETED"
store.put_executions([fe_exec])
print("Feature engineering step completed.")

# 5.3 Model Training
print("\n--- 5.3 Model Training ---")
train_exec = metadata_store_pb2.Execution()
train_exec.type_id = trainer_type_id
train_exec.properties["state"].string_value = "RUNNING"
train_exec.properties["learning_rate"].double_value = 0.05
train_exec.properties["n_estimators"].int_value = 300
[train_exec_id] = store.put_executions([train_exec])
print(f"Created ModelTrainer execution (id={train_exec_id})")

train_input_event = metadata_store_pb2.Event()
train_input_event.artifact_id = player_features_id
train_input_event.execution_id = train_exec_id
train_input_event.type = metadata_store_pb2.Event.DECLARED_INPUT
store.put_events([train_input_event])

trained_model = metadata_store_pb2.Artifact()
trained_model.uri = "gs://sports-ml-models/bundesliga/valuation_xgb_v1.pkl"
trained_model.type_id = model_type_id
trained_model.properties["model_type"].string_value = "XGBRegressor"
trained_model.properties["version"].int_value = 1
trained_model.properties["framework"].string_value = "xgboost-2.0.3"
[model_artifact_id] = store.put_artifacts([trained_model])
print(f"Created ValuationModel artifact (id={model_artifact_id})")

train_output_event = metadata_store_pb2.Event()
train_output_event.artifact_id = model_artifact_id
train_output_event.execution_id = train_exec_id
train_output_event.type = metadata_store_pb2.Event.DECLARED_OUTPUT
store.put_events([train_output_event])

train_exec.id = train_exec_id
train_exec.properties["state"].string_value = "COMPLETED"
store.put_executions([train_exec])
print("Model training step completed.")

# 5.4 Model Evaluation
print("\n--- 5.4 Model Evaluation ---")
eval_exec = metadata_store_pb2.Execution()
eval_exec.type_id = evaluator_type_id
eval_exec.properties["state"].string_value = "RUNNING"
eval_exec.properties["eval_split"].string_value = "test_20pct"
[eval_exec_id] = store.put_executions([eval_exec])
print(f"Created ModelEvaluator execution (id={eval_exec_id})")

eval_model_input = metadata_store_pb2.Event()
eval_model_input.artifact_id = model_artifact_id
eval_model_input.execution_id = eval_exec_id
eval_model_input.type = metadata_store_pb2.Event.DECLARED_INPUT

eval_features_input = metadata_store_pb2.Event()
eval_features_input.artifact_id = player_features_id
eval_features_input.execution_id = eval_exec_id
eval_features_input.type = metadata_store_pb2.Event.DECLARED_INPUT

store.put_events([eval_model_input, eval_features_input])

eval_metrics = metadata_store_pb2.Artifact()
eval_metrics.uri = "gs://sports-ml-models/bundesliga/eval_metrics_v1.json"
eval_metrics.type_id = eval_type_id
eval_metrics.properties["rmse"].double_value = 3.42
eval_metrics.properties["mae"].double_value = 2.15
eval_metrics.properties["r2_score"].double_value = 0.847
[eval_metrics_id] = store.put_artifacts([eval_metrics])
print(f"Created EvaluationMetrics artifact (id={eval_metrics_id})")

eval_output_event = metadata_store_pb2.Event()
eval_output_event.artifact_id = eval_metrics_id
eval_output_event.execution_id = eval_exec_id
eval_output_event.type = metadata_store_pb2.Event.DECLARED_OUTPUT
store.put_events([eval_output_event])

eval_exec.id = eval_exec_id
eval_exec.properties["state"].string_value = "COMPLETED"
store.put_executions([eval_exec])
print("Model evaluation step completed.")
print(f"\nResults: RMSE={3.42}, MAE={2.15}, R2={0.847}")

# ============================================================
# 6. Group Under a Context
# ============================================================
print("\n" + "=" * 60)
print("6. GROUP UNDER CONTEXT: SEASON EXPERIMENT")
print("=" * 60)

season_exp_type = metadata_store_pb2.ContextType()
season_exp_type.name = "SeasonExperiment"
season_exp_type.properties["league"] = metadata_store_pb2.STRING
season_exp_type.properties["season"] = metadata_store_pb2.STRING
season_exp_type.properties["notes"] = metadata_store_pb2.STRING
season_exp_type_id = store.put_context_type(season_exp_type)
print(f"Registered ContextType 'SeasonExperiment' with id={season_exp_type_id}")

buli_exp = metadata_store_pb2.Context()
buli_exp.type_id = season_exp_type_id
buli_exp.name = "bundesliga_2023_24_exp1"
buli_exp.properties["league"].string_value = "Bundesliga"
buli_exp.properties["season"].string_value = "2023-24"
buli_exp.properties["notes"].string_value = "Baseline XGBoost valuation model using per-90 features."
[buli_exp_id] = store.put_contexts([buli_exp])
print(f"Created context 'bundesliga_2023_24_exp1' (id={buli_exp_id})")

artifact_ids_to_attribute = [raw_data_id, player_features_id, model_artifact_id, eval_metrics_id]
attributions = []
for aid in artifact_ids_to_attribute:
    attr = metadata_store_pb2.Attribution()
    attr.artifact_id = aid
    attr.context_id = buli_exp_id
    attributions.append(attr)

exec_ids_to_associate = [ingest_exec_id, fe_exec_id, train_exec_id, eval_exec_id]
associations = []
for eid in exec_ids_to_associate:
    assoc = metadata_store_pb2.Association()
    assoc.execution_id = eid
    assoc.context_id = buli_exp_id
    associations.append(assoc)

store.put_attributions_and_associations(attributions, associations)
print(f"Linked {len(attributions)} artifacts and {len(associations)} executions to context.")

# ============================================================
# 7. Querying the Metadata Store
# ============================================================
print("\n" + "=" * 60)
print("7. QUERYING THE METADATA STORE")
print("=" * 60)

# 7a
print("\n--- 7a. All ValuationModel artifacts ---")
all_models = store.get_artifacts_by_type("ValuationModel")
for m in all_models:
    print(f"  id={m.id}, uri={m.uri}")
    print(f"    model_type={m.properties['model_type'].string_value}")
    print(f"    version={m.properties['version'].int_value}")
    print(f"    framework={m.properties['framework'].string_value}")

# 7b
print("\n--- 7b. Artifacts in 'bundesliga_2023_24_exp1' ---")
ctx_artifacts = store.get_artifacts_by_context(buli_exp_id)
for a in ctx_artifacts:
    a_type = store.get_artifact_types_by_id([a.type_id])[0]
    print(f"  [{a_type.name}] id={a.id}, uri={a.uri}")

print("\nExecutions in 'bundesliga_2023_24_exp1':")
ctx_executions = store.get_executions_by_context(buli_exp_id)
for e in ctx_executions:
    e_type = store.get_execution_types_by_id([e.type_id])[0]
    print(f"  [{e_type.name}] id={e.id}, state={e.properties['state'].string_value}")

# 7c
print("\n--- 7c. Lineage trace: What produced the ValuationModel? ---")
model_events = store.get_events_by_artifact_ids([model_artifact_id])
for event in model_events:
    event_type_str = metadata_store_pb2.Event.Type.Name(event.type)
    print(f"  Event: artifact_id={event.artifact_id}, "
          f"execution_id={event.execution_id}, type={event_type_str}")

train_events = store.get_events_by_execution_ids([train_exec_id])
print(f"\nEvents for training execution (id={train_exec_id}):")
for event in train_events:
    event_type_str = metadata_store_pb2.Event.Type.Name(event.type)
    artifact = store.get_artifacts_by_id([event.artifact_id])[0]
    print(f"  artifact_id={event.artifact_id} ({artifact.uri}), type={event_type_str}")

# 7d
print("\n--- 7d. Full upstream lineage of the trained model ---")

def trace_upstream(store, artifact_id, depth=0):
    artifact = store.get_artifacts_by_id([artifact_id])[0]
    a_type = store.get_artifact_types_by_id([artifact.type_id])[0]
    indent = "  " * depth
    print(f"{indent}[{a_type.name}] id={artifact.id}, uri={artifact.uri}")
    events = store.get_events_by_artifact_ids([artifact_id])
    for event in events:
        if event.type == metadata_store_pb2.Event.DECLARED_OUTPUT:
            exec_id = event.execution_id
            execution = store.get_executions_by_id([exec_id])[0]
            e_type = store.get_execution_types_by_id([execution.type_id])[0]
            print(f"{indent}  <- produced by [{e_type.name}] (exec_id={exec_id})")
            exec_events = store.get_events_by_execution_ids([exec_id])
            for e_event in exec_events:
                if e_event.type == metadata_store_pb2.Event.DECLARED_INPUT:
                    trace_upstream(store, e_event.artifact_id, depth + 2)

trace_upstream(store, model_artifact_id)

# 7e
print("\n--- 7e. Conditional queries ---")
print("Completed executions of type 'ModelTrainer':")
completed_trainers = store.get_executions(
    list_options=mlmd.ListOptions(
        filter_query='type = "ModelTrainer" AND properties.state.string_value = "COMPLETED"'
    )
)
for ex in completed_trainers:
    print(f"  exec_id={ex.id}, lr={ex.properties['learning_rate'].double_value}, "
          f"n_est={ex.properties['n_estimators'].int_value}")

print("\nArtifacts with URI containing 'bundesliga':")
buli_artifacts = store.get_artifacts(
    list_options=mlmd.ListOptions(
        filter_query='uri LIKE "%bundesliga%"'
    )
)
for a in buli_artifacts:
    a_type = store.get_artifact_types_by_id([a.type_id])[0]
    print(f"  [{a_type.name}] id={a.id}, uri={a.uri}")

# ============================================================
# 8. Pipeline Run 2: Updated Feature Engineering
# ============================================================
print("\n" + "=" * 60)
print("8. PIPELINE RUN 2: UPDATED FEATURE ENGINEERING")
print("=" * 60)

fe_exec_v2 = metadata_store_pb2.Execution()
fe_exec_v2.type_id = fe_type_id
fe_exec_v2.properties["state"].string_value = "COMPLETED"
fe_exec_v2.properties["aggregation_method"].string_value = "per_90_weighted_by_minutes"
[fe_exec_v2_id] = store.put_executions([fe_exec_v2])

fe_v2_input = metadata_store_pb2.Event()
fe_v2_input.artifact_id = raw_data_id
fe_v2_input.execution_id = fe_exec_v2_id
fe_v2_input.type = metadata_store_pb2.Event.DECLARED_INPUT
store.put_events([fe_v2_input])

player_features_v2 = metadata_store_pb2.Artifact()
player_features_v2.uri = "gs://sports-ml-data/bundesliga/2023-24/player_features_v2.parquet"
player_features_v2.type_id = feature_set_type_id
player_features_v2.properties["num_players"].int_value = 523
player_features_v2.properties["num_features"].int_value = 58
player_features_v2.properties["feature_version"].string_value = "v2.0"
[player_features_v2_id] = store.put_artifacts([player_features_v2])

fe_v2_output = metadata_store_pb2.Event()
fe_v2_output.artifact_id = player_features_v2_id
fe_v2_output.execution_id = fe_exec_v2_id
fe_v2_output.type = metadata_store_pb2.Event.DECLARED_OUTPUT
store.put_events([fe_v2_output])

print(f"Feature engineering v2 completed: 58 features (was 42 in v1)")

train_exec_v2 = metadata_store_pb2.Execution()
train_exec_v2.type_id = trainer_type_id
train_exec_v2.properties["state"].string_value = "COMPLETED"
train_exec_v2.properties["learning_rate"].double_value = 0.03
train_exec_v2.properties["n_estimators"].int_value = 500
[train_exec_v2_id] = store.put_executions([train_exec_v2])

train_v2_input = metadata_store_pb2.Event()
train_v2_input.artifact_id = player_features_v2_id
train_v2_input.execution_id = train_exec_v2_id
train_v2_input.type = metadata_store_pb2.Event.DECLARED_INPUT
store.put_events([train_v2_input])

model_v2 = metadata_store_pb2.Artifact()
model_v2.uri = "gs://sports-ml-models/bundesliga/valuation_xgb_v2.pkl"
model_v2.type_id = model_type_id
model_v2.properties["model_type"].string_value = "XGBRegressor"
model_v2.properties["version"].int_value = 2
model_v2.properties["framework"].string_value = "xgboost-2.0.3"
[model_v2_id] = store.put_artifacts([model_v2])

train_v2_output = metadata_store_pb2.Event()
train_v2_output.artifact_id = model_v2_id
train_v2_output.execution_id = train_exec_v2_id
train_v2_output.type = metadata_store_pb2.Event.DECLARED_OUTPUT
store.put_events([train_v2_output])

eval_exec_v2 = metadata_store_pb2.Execution()
eval_exec_v2.type_id = evaluator_type_id
eval_exec_v2.properties["state"].string_value = "COMPLETED"
eval_exec_v2.properties["eval_split"].string_value = "test_20pct"
[eval_exec_v2_id] = store.put_executions([eval_exec_v2])

store.put_events([
    metadata_store_pb2.Event(
        artifact_id=model_v2_id, execution_id=eval_exec_v2_id,
        type=metadata_store_pb2.Event.DECLARED_INPUT),
    metadata_store_pb2.Event(
        artifact_id=player_features_v2_id, execution_id=eval_exec_v2_id,
        type=metadata_store_pb2.Event.DECLARED_INPUT)
])

eval_metrics_v2 = metadata_store_pb2.Artifact()
eval_metrics_v2.uri = "gs://sports-ml-models/bundesliga/eval_metrics_v2.json"
eval_metrics_v2.type_id = eval_type_id
eval_metrics_v2.properties["rmse"].double_value = 2.87
eval_metrics_v2.properties["mae"].double_value = 1.73
eval_metrics_v2.properties["r2_score"].double_value = 0.912
[eval_metrics_v2_id] = store.put_artifacts([eval_metrics_v2])

store.put_events([metadata_store_pb2.Event(
    artifact_id=eval_metrics_v2_id, execution_id=eval_exec_v2_id,
    type=metadata_store_pb2.Event.DECLARED_OUTPUT)])

print(f"Pipeline Run 2 completed.")
print(f"Model v2 results: RMSE=2.87, MAE=1.73, R2=0.912")

# Group run 2 under a new context
buli_exp2 = metadata_store_pb2.Context()
buli_exp2.type_id = season_exp_type_id
buli_exp2.name = "bundesliga_2023_24_exp2"
buli_exp2.properties["league"].string_value = "Bundesliga"
buli_exp2.properties["season"].string_value = "2023-24"
buli_exp2.properties["notes"].string_value = "v2 features with progressive carries + pressing. Lower LR, more trees."
[buli_exp2_id] = store.put_contexts([buli_exp2])

new_attrs = []
for aid in [player_features_v2_id, model_v2_id, eval_metrics_v2_id]:
    a = metadata_store_pb2.Attribution()
    a.artifact_id = aid
    a.context_id = buli_exp2_id
    new_attrs.append(a)

new_assocs = []
for eid in [fe_exec_v2_id, train_exec_v2_id, eval_exec_v2_id]:
    a = metadata_store_pb2.Association()
    a.execution_id = eid
    a.context_id = buli_exp2_id
    new_assocs.append(a)

store.put_attributions_and_associations(new_attrs, new_assocs)
print(f"Linked run 2 artifacts/executions to context 'bundesliga_2023_24_exp2'.")

# ============================================================
# 9. Compare Experiments
# ============================================================
print("\n" + "=" * 60)
print("9. EXPERIMENT COMPARISON: Bundesliga 2023-24")
print("=" * 60)

all_contexts = store.get_contexts_by_type("SeasonExperiment")
for ctx in all_contexts:
    print(f"\nExperiment: {ctx.name}")
    print(f"  Notes: {ctx.properties['notes'].string_value}")

    ctx_artifacts = store.get_artifacts_by_context(ctx.id)
    for art in ctx_artifacts:
        art_type = store.get_artifact_types_by_id([art.type_id])[0]
        if art_type.name == "EvaluationMetrics":
            print(f"  RMSE:  {art.properties['rmse'].double_value}")
            print(f"  MAE:   {art.properties['mae'].double_value}")
            print(f"  R2:    {art.properties['r2_score'].double_value}")
        elif art_type.name == "PlayerFeatureSet":
            print(f"  Features: {art.properties['num_features'].int_value} "
                  f"(version {art.properties['feature_version'].string_value})")

    ctx_execs = store.get_executions_by_context(ctx.id)
    for ex in ctx_execs:
        ex_type = store.get_execution_types_by_id([ex.type_id])[0]
        if ex_type.name == "ModelTrainer":
            print(f"  Hyperparams: lr={ex.properties['learning_rate'].double_value}, "
                  f"n_estimators={ex.properties['n_estimators'].int_value}")

# ============================================================
# 10. Summary
# ============================================================
print("\n" + "=" * 60)
print("10. METADATA STORE SUMMARY")
print("=" * 60)

all_artifacts = store.get_artifacts()
all_executions = store.get_executions()
all_contexts = store.get_contexts()
all_events = store.get_events_by_artifact_ids([a.id for a in all_artifacts])

print(f"  Artifact types:  {len(store.get_artifact_types())}")
print(f"  Execution types: {len(store.get_execution_types())}")
print(f"  Context types:   {len(store.get_context_types())}")
print(f"  Artifacts:       {len(all_artifacts)}")
print(f"  Executions:      {len(all_executions)}")
print(f"  Contexts:        {len(all_contexts)}")
print(f"  Events:          {len(all_events)}")

1. SETUP
ml-metadata imported successfully.

Metadata store initialised successfully (in-memory SQLite backend).

3. REGISTER ARTIFACT TYPES
Registered ArtifactType 'MatchEventData' with id=10
Registered ArtifactType 'PlayerFeatureSet' with id=11
Registered ArtifactType 'ValuationModel' with id=12
Registered ArtifactType 'EvaluationMetrics' with id=13

Total registered artifact types: 4
  - MatchEventData (id=10): properties={'num_matches': 1, 'league': 3, 'season': 3}
  - PlayerFeatureSet (id=11): properties={'num_players': 1, 'num_features': 1, 'feature_version': 3}
  - ValuationModel (id=12): properties={'version': 1, 'framework': 3, 'model_type': 3}
  - EvaluationMetrics (id=13): properties={'rmse': 2, 'mae': 2, 'r2_score': 2}

4. REGISTER EXECUTION TYPES
Registered ExecutionType 'DataIngestor' with id=14
Registered ExecutionType 'FeatureEngineer' with id=15
Registered ExecutionType 'ModelTrainer' with id=16
Registered ExecutionType 'ModelEvaluator' with id=17

Total registered exe

## Key Takeaways

1. **Artifact Types** define the schema for data objects (datasets, features, models, metrics) flowing through a pipeline.
2. **Execution Types** represent the pipeline steps/components (ingestor, feature engineer, trainer, evaluator).
3. **Events** capture the input/output relationships between artifacts and executions, enabling lineage tracing.
4. **Contexts** group related artifacts and executions into logical experiments, making it easy to compare runs.
5. **Filter queries** allow conditional retrieval of artifacts and executions based on property values.
6. By recursively following events, we can trace the **full provenance** of any artifact back to the original data source.

---

**References:**
- [ML Metadata Guide (TensorFlow)](https://www.tensorflow.org/tfx/guide/mlmd)
- [MLMD API Documentation](https://www.tensorflow.org/tfx/ml_metadata/api_docs/python/mlmd)
- [Original Lab Repo](https://github.com/raminmohammadi/MLOps/tree/main/Labs/MLMD_Labs)